In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com" 
# os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import BertForQuestionAnswering
from transformers import TrainingArguments, Trainer
import torch
import torch.nn.functional as F
from pathlib import Path
from datasets import Dataset

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


# Sentiment Analysis
- Given some text, is it positive or negative?

In [2]:
classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://hf-mirror.com/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


In [3]:
sentences = [
    "This movie was amazing!",
    "We hope you don't hate it.",
    "This was a really bad movie!"
]

In [4]:
results = classifier(sentences)

for sentence, result in zip(sentences, results):
    print("Sentence: {} \t\t Result: {}".format(sentence, result))

Sentence: This movie was amazing! 		 Result: {'label': 'POSITIVE', 'score': 0.9998800754547119}
Sentence: We hope you don't hate it. 		 Result: {'label': 'NEGATIVE', 'score': 0.5308576226234436}
Sentence: This was a really bad movie! 		 Result: {'label': 'NEGATIVE', 'score': 0.9997768998146057}


# Question Answering
- Given some context, answer a question.

In [16]:
context = (
    "Lorem Ipsum is simply dummy text of the printing and typesetting industry. "
    "Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took "
    "a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but "
    "also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the "
    "1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop "
    "publishing software like Aldus PageMaker including versions of Lorem Ipsum."
)

In [17]:
questions = [
    "What is Lorem Ipsum?",
    "Since when was it the indutry standard?",
    "How did Lorem Ipsum come to be?",
    "When did it become popular again?",
    "When were Letraset sheets cotaining Lorem Ipsum put out?"
]

In [23]:
# Distilled BERT
question_answering = pipeline("question-answering", model="deepset/roberta-base-squad2", device=-1)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [32]:
question_answering({
    'question' : questions[0], 
    'context' : context
})

{'score': 0.4210926294326782,
 'start': 22,
 'end': 73,
 'answer': 'dummy text of the printing and typesetting industry'}

In [21]:
print(question_answering.model.name_or_path)

distilbert/distilbert-base-cased-distilled-squad


In [11]:
# T5
text2text_generator = pipeline("text2text-generation", device=-1)

No model was supplied, defaulted to google-t5/t5-base and revision a9723ea (https://hf-mirror.com/google-t5/t5-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
'(ReadTimeoutError("HTTPSConnectionPool(host='hf-mirror.com', port=443): Read timed out. (read timeout=10)"), '(Request ID: ebf3f93d-acda-4ac7-bf03-a56ad3a8bfd3)')' thrown while requesting HEAD https://hf-mirror.com/google-t5/t5-base/resolve/a9723ea/config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='hf-mirror.com', port=443): Read timed out. (read timeout=10)"), '(Request ID: a0624fc7-b847-4d0e-a10f-a9792c4415d4)')' thrown while requesting HEAD https://hf-mirror.com/google-t5/t5-base/resolve/a9723ea/config.json
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='hf-mirror.com', port=443): Read timed out. (read timeout=10)"), '(Request ID: 20e94cd2-1641-452c-8ba0-269e6dfc2dd6)')' thrown while requesting HEAD https://hf-m

In [13]:
text2text_generator("question: Where is the Keio University main campus located? context: Keio University is a private university in Japan. Its main campus is located in Mita, Minato, Tokyo.")

[{'generated_text': 'Mita, Minato, Tokyo'}]

# Summarization
- Given some long text, write a short summary.

In [33]:
context = (
    "Lorem Ipsum is simply dummy text of the printing and typesetting industry. "
    "Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took "
    "a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but "
    "also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the "
    "1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop "
    "publishing software like Aldus PageMaker including versions of Lorem Ipsum."
)

In [35]:
summarizer = pipeline("summarization", max_length=60, device=-1)

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://hf-mirror.com/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [39]:
summarizer(context, max_length=60, min_length=5, do_sample=False)

[{'summary_text': " Lorem Ipsum has been the industry's standard dummy text ever since the 1500s . It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages ."}]

# Text Generation
- Generate some text given some context or from scratch.

In [41]:
generator = pipeline(model="gpt2", device=-1)

Device set to use cpu


In [42]:
# These parameters will return suggestions, and only the newly created text making it easier for prompting suggestions.
generator("I love going to the", num_return_sequences=4, return_full_text=True)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'I love going to the office to talk to people. I love meeting new people. I am extremely lucky to be able to do that."\n\nWhile it was initially reported that Flynn had left his first job as national security adviser in February, sources close to the situation tell Fox News that he has returned to the White House.\n\nThe source said Flynn told the White House that he felt he was not being asked to resign, at least for now. He indicated that he did not know the president had been asked to resign, though he did not deny he had made a mistake.\n\n"I didn\'t think he had that conversation. I didn\'t know that he was going to resign," the source added.\n\nFlynn\'s departure was confirmed by White House spokesman Sean Spicer on Tuesday afternoon.\n\nThe White House confirmed to Fox News on Thursday that Flynn would not be staying at the White House.\n\nFlynn resigned from his position on an intelligence committee in February after a meeting with the Russian ambassador. He